## Setup


In [ ]:
%pip install -q medmnist umap-learn scikit-learn seaborn

In [ ]:
import os, json, hashlib, warnings
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from IPython.display import display
import medmnist
from medmnist import INFO, DermaMNIST

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Pastas (ajuste se necessário; no Colab "./data" e "./reports" ficam na sessão)
DATA_DIR = os.path.abspath("./data")
OUT_DIR  = os.path.abspath("./reports")
FIG_DIR  = os.path.join(OUT_DIR, "figures")
for d in (DATA_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)

# Estilo: dpi baixo na tela (notebook leve) e 300 dpi apenas ao salvar
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 100, 'savefig.dpi': 300})

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name), bbox_inches='tight')

SUMMARY = {}   # acumula os números que irão para o relatório

print(f"medmnist {medmnist.__version__} | torch {torch.__version__} | numpy {np.__version__} | device {DEVICE} | seed {SEED}")
SUMMARY['versoes'] = dict(medmnist=medmnist.__version__, torch=torch.__version__, numpy=np.__version__, seed=SEED)

Análise do Balanceamento de Classes

In [ ]:
labels = [int(x) for x in train_dataset.labels.squeeze()]
counts = Counter(labels)
sorted_counts = [counts[i] for i in range(7)]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d9534f' if i == 5 else '#2c3e50' for i in range(7)] # Destaca Melanoma em vermelho

bars = ax.bar([f"C{i}\n{derma_classes[i].split('(')[1][:-1]}" for i in range(7)], sorted_counts, color=colors, alpha=0.85)
ax.set_title("DermaMNIST: Distribuição das Classes (Treino)", fontsize=11, fontweight='bold')
ax.set_ylabel("Contagem Absoluta")

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 80, f'{height}', ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, 5200)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "eda_dermamnist_imbalance.png"), dpi=300, bbox_inches='tight')
plt.show()

ir = max(sorted_counts) / min(sorted_counts)
print(f"=== Métrica de Assimetria ===")
print(f"Classe Dominante (C4 - nv): {counts[4]} amostras")
print(f"Classe Rara (C3 - df): {counts[3]} amostras")
print(f"Imbalance Ratio (IR): {ir:.2f} (Extremamente desbalanceado!)")

Painel Clínico Visual (7 Classes de Lesões Dermatológicas)

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(14, 2.5))
fig.suptitle("Amostra Diagnóstica por Classe (DermaMNIST)", fontsize=11, fontweight='bold', y=1.05)

found = {}
for img, lbl in train_dataset:
    l = int(lbl)
    if l not in found:
        found[l] = img
    if len(found) == 7:
        break

for i in range(7):
    img_np = found[i].permute(1, 2, 0).numpy()
    axes[i].imshow(img_np)
    axes[i].set_title(f"Class {i}\n{derma_classes[i].split('(')[1][:-1].upper()}", fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "eda_dermamnist_samples.png"), dpi=300, bbox_inches='tight')
plt.show()

Visualização da Grade de Patches $4 \times 4$ e Extração de Média/Std

In [ ]:
# Overlay do Fatiamento de Patches
sample_img, _ = train_dataset[12]
img_np = sample_img.permute(1, 2, 0).numpy()

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(img_np)

for x in range(0, 32, 4):
    ax.axvline(x - 0.5, color='yellow', linestyle=':', linewidth=0.8)
    ax.axhline(x - 0.5, color='yellow', linestyle=':', linewidth=0.8)

ax.set_title("Grade de Patches 4x4 sobre Lesão Cutânea", fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "eda_patch_grid_derma.png"), dpi=300, bbox_inches='tight')
plt.show()

# Normalização Específica
loader = torch.utils.data.DataLoader(train_dataset, batch_size=500, shuffle=False)
mean, std, total = torch.zeros(3), torch.zeros(3), 0

for images, _ in loader:
    b = images.size(0)
    images = images.view(b, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total += b

mean /= total
std /= total

print("=== Parâmetros para PyTorch Normalization (DermaMNIST) ===")
print(f"mean = [{mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f}]")
print(f"std  = [{std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f}]")